# 08e · Predict the test complaints with the trained RoBERTa (on Colab): BankSight

**Run this notebook on Google Colab.** Choose **Runtime → Change runtime type → T4 GPU** first.

**What it does:** loads the RoBERTa you fine-tuned in 08b (saved in Drive), reads `test.parquet` from 08d, predicts the probability of relief for each
test complaint, and saves them to a CSV.

**What this does:** checks that a GPU is attached.

In [ ]:
import torch

print("GPU available:", torch.cuda.is_available())

**What this does:** installs the `transformers` library if Colab does not have it.

In [ ]:
# %pip install -q transformers

**What this does:** loads the libraries.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from transformers import AutoModelForSequenceClassification, AutoTokenizer

**What this does:** connects Google Drive (falls back to the local folder when not on Colab) and checks the files exist.

In [ ]:
try:
    from google.colab import drive
    drive.mount("/content/drive")
    DATA_DIR = Path("/content/drive/MyDrive/banksight_colab")
except ImportError:
    DATA_DIR = Path("../data/colab")

OUT_DIR = DATA_DIR / "outputs"
MODEL_TAG = "roberta"
MAX_LENGTH = 256     # the same value used in training
BATCH_SIZE = 64

assert (DATA_DIR / "test.parquet").exists(), "Upload test.parquet (from 08d) to the banksight_colab folder."
assert (OUT_DIR / f"{MODEL_TAG}_relief").exists(), "The fine-tuned model folder from 08b is missing."
print("files found")

**What this does:** loads the test text and the fine-tuned model.

In [ ]:
test = pd.read_parquet(DATA_DIR / "test.parquet")
print("columns:", list(test.columns), "| rows:", len(test))

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
tokenizer = AutoTokenizer.from_pretrained(OUT_DIR / f"{MODEL_TAG}_relief")
model = AutoModelForSequenceClassification.from_pretrained(OUT_DIR / f"{MODEL_TAG}_relief")
model.to(device)
model.eval()
use_gpu = torch.cuda.is_available()

**What this does:** predicts in batches (no learning happens). The model's score is turned into a probability with the sigmoid function.

In [ ]:
probabilities = []

with torch.no_grad():
    for start in range(0, len(test), BATCH_SIZE):
        texts = test["input_text"].iloc[start:start + BATCH_SIZE].tolist()
        batch = tokenizer(texts, padding=True, truncation=True, max_length=MAX_LENGTH, return_tensors="pt").to(device)
        with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=use_gpu):
            scores = model(**batch).logits.squeeze(-1)
        probabilities.extend(torch.sigmoid(scores.float()).cpu().numpy().tolist())

probabilities = np.array(probabilities)
print("predicted", len(probabilities), "complaints | average probability", round(probabilities.mean(), 3))

**What this does:** saves the probabilities. Download this file into your laptop's `data/colab/outputs/` folder.

In [ ]:
result = pd.DataFrame({"complaint_id": test["complaint_id"], "p_relief": probabilities})
result.to_csv(OUT_DIR / f"test_predictions_{MODEL_TAG}.csv", index=False)
print("saved", OUT_DIR / f"test_predictions_{MODEL_TAG}.csv")

## How to explain this in an interview
> "Inference is separate from training: the same saved model, the same text format and the same maximum length, run once on the sealed test text. The GPU never saw the test outcomes."